# 06 · Data Cleaning

Checks every acquired image (PlantVillage, FieldPlant, Tanzania potato and PlantDoc)
and records which ones to keep in `metadata/clean_manifest.csv`. **Nothing is deleted
from Drive**; the split reads the manifest.

| Rule | Reason recorded |
|---|---|
| Does not open or decode fully | `unreadable` |
| Either side below `config.MIN_IMAGE_SIZE` | `too_small` |
| Same file in two different classes (all copies dropped) | `duplicate_conflicting_labels` |
| Repeat of a file in the same class (first copy kept) | `exact_duplicate` |
| PlantDoc image that is also a training image | `in_training_sources` |
| PlantDoc image that is a near-copy of a training image | `near_duplicate_of_training` |

Near-duplicates inside the training pool (perceptual hash within
`config.NEAR_DUPLICATE_MAX_DISTANCE` bits) are kept but share a `dup_group`, so the
split can keep them in one partition. Runs on a CPU runtime.

In [ ]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

Mounted at /content/drive


In [ ]:
# Keep the Drive copy of the repository in sync with GitHub.
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (1/1), 950 bytes | 45.00 KiB/s, done.
From https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis
   bdd0170..d3c080c  main       -> origin/main
Updating bdd0170..d3c080c
error: Your local changes to the following files would be overwritten by merge:
	ml/notebooks/06_image_cleaning.ipynb
Please commit your changes or stash them before you merge.
Aborting
## main...origin/main [behind 2]
 M ml/notebooks/06_image_cleaning.ipynb


In [ ]:
%pip install -q -r {ML_DIR}/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.3 MB/s eta 0:00:00


## List every image

In [ ]:
import pandas as pd

from src.data import cleaning

images = pd.DataFrame(
    cleaning.collect(config.PLANTVILLAGE_DIR, "plantvillage", cleaning.TRAIN_POOL, config.PLANTVILLAGE_CLASSES)
    + cleaning.collect(config.FIELDPLANT_DIR, "fieldplant", cleaning.TRAIN_POOL)
    + cleaning.collect(config.TANZANIA_DIR, "tanzania", cleaning.TRAIN_POOL)
    + cleaning.collect(config.PLANTDOC_DIR / "in_scope", "plantdoc", "ood_in_scope")
    + cleaning.collect(config.PLANTDOC_DIR / "unseen", "plantdoc", "ood_unseen")
)
display(images.groupby(["subset", "source"]).size().rename("images").reset_index())
print("Total:", len(images))

,subset,source,images
0,ood_in_scope,plantdoc,970
1,ood_unseen,plantdoc,374
2,train_pool,fieldplant,889
3,train_pool,plantvillage,12631
4,train_pool,tanzania,3000


Total: 17864


## Check every image

Opens and fully decodes each image, and computes its file hash (exact duplicates) and
perceptual hash (near-duplicates). Reading from Drive is the slow part; expect 10–20 minutes.

In [ ]:
checks = pd.DataFrame(cleaning.check_all(images["path"].tolist()))
checked = pd.concat([images, checks], axis=1)
print("Unreadable:", checked["error"].notna().sum())
display(checked.loc[checked["error"].notna(), ["source", "class", "path", "error"]].head(20))

Checking images:   0%|          | 0/17864 [00:00<?, ?it/s]

Unreadable: 0


,source,class,path,error


## Apply the cleaning rules

In [ ]:
manifest = cleaning.clean(checked, config.MIN_IMAGE_SIZE, config.NEAR_DUPLICATE_MAX_DISTANCE)

summary = (
    manifest.assign(reason=manifest["reason"].replace("", "kept"))
    .pivot_table(index=["subset", "source"], columns="reason", values="path", aggfunc="size", fill_value=0)
)
display(summary)

pool = manifest[manifest["dup_group"].notna()]
group_sizes = pool.groupby("dup_group").size()
mixed = pool.groupby("dup_group")["class"].nunique()
print("Kept:", manifest["keep"].sum(), "of", len(manifest))
print("Near-duplicate groups (2+ images):", (group_sizes > 1).sum(),
      "| images in them:", group_sizes[group_sizes > 1].sum(),
      "| largest group:", group_sizes.max())
print("Groups mixing classes:", (mixed > 1).sum())

reason                     duplicate_conflicting_labels  exact_duplicate  \
subset       source                                                        
ood_in_scope plantdoc                                17                0   
ood_unseen   plantdoc                                 1                2   
train_pool   fieldplant                               0                0   
             plantvillage                             0               14   
             tanzania                                 0              203   

reason                      kept  
subset       source               
ood_in_scope plantdoc        953  
ood_unseen   plantdoc        371  
train_pool   fieldplant      889  
             plantvillage  12617  
             tanzania       2797

Kept: 17627 of 17864
Near-duplicate groups (2+ images): 91 | images in them: 218 | largest group: 9
Groups mixing classes: 0


## Save the manifest

`clean_manifest.csv` has one row per image with its keep flag, reason, hashes and
`dup_group` (paths relative to the raw data folder); `cleaning_summary.csv` holds the
table above for the data report.

In [ ]:
out = manifest.drop(columns="error").copy()
out["path"] = [Path(p).relative_to(config.RAW_DATA_DIR).as_posix() for p in out["path"]]
out.to_csv(config.METADATA_DIR / "clean_manifest.csv", index=False)
summary.to_csv(config.METADATA_DIR / "cleaning_summary.csv")
print("Saved clean_manifest.csv and cleaning_summary.csv to", config.METADATA_DIR)

Saved clean_manifest.csv and cleaning_summary.csv to /content/drive/MyDrive/CropDiseaseDiagnosis/datasets/metadata


In [ ]:
import shutil
nb = REPO_DIR / "ml/notebooks/06_image_cleaning.ipynb"
shutil.copy(nb, "/content/06_backup.ipynb")      # keep today's run safe

!git -C {REPO_DIR} checkout -- ml/notebooks/06_image_cleaning.ipynb
!git -C {REPO_DIR} pull --ff-only

shutil.copy("/content/06_backup.ipynb", nb)       # put today's run back
!git -C {REPO_DIR} status --short --branch

In [9]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount("/content/drive")
print("Drive synced and remounted.")

Mounted at /content/drive
Drive synced and remounted.


## Save to GitHub

Commits the changed notebook to a new branch and prints a link to open the pull
request (`main` is protected). Change the branch name if this cell has already been
run once. Save the notebook (Ctrl+S) before running this cell.

In [ ]:
from google.colab import userdata

from src import github_sync

print(github_sync.status(REPO_DIR))
print()
print(github_sync.push(
    REPO_DIR,
    branch="colab/image-cleaning",
    message="Clean the acquired images and save the manifest",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
))